<a href="https://colab.research.google.com/github/JakeOh/202605_BD57/blob/main/lab_python/da12_shape.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DataFrame의 모양(shape) 변경

wide(column이 많은) DF <---> long(row이 많은) DF

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

# stack vs unstack

In [2]:
df = pd.DataFrame(data=np.arange(1, 7).reshape((2, 3)),
                  columns=['a', 'b', 'c'],
                  index=['X', 'Y'])
df

,a,b,c
X,1,2,3
Y,4,5,6


In [3]:
df_stacked = df.stack()  #> 컬럼의 이름들이 행 인덱스로 설정. stack: column --> row
df_stacked

X  a    1
   b    2
   c    3
Y  a    4
   b    5
   c    6
dtype: int64

In [4]:
df_stacked.index.nlevels

2

In [7]:
df_unstack1 = df_stacked.unstack()  #> 행 인덱스가 컬럼 이름으로 변환. row ---> column
df_unstack1

,a,b,c
X,1,2,3
Y,4,5,6


In [9]:
df_unstack2 = df_stacked.unstack(level=0)  # 첫번째 레벨의 행 인덱스가 컬럼 이름으로 변환. row ---> column
df_unstack2

,X,Y
a,1,4
b,2,5
c,3,6


## 컬럼 이름이 계층적(다중) 인덱스인 경우

In [10]:
df = pd.DataFrame(data=np.arange(1, 13).reshape((2, 6)),
                  columns=[['Fri', 'Fri', 'Sat', 'Sat', 'Sun', 'Sun'],
                           ['Lunch', 'Dinner'] * 3])
df

Fri          Sat          Sun       
  Lunch Dinner Lunch Dinner Lunch Dinner
0     1      2     3      4     5      6
1     7      8     9     10    11     12

In [11]:
df.columns  #> MultiIndex - 튜플들의 배열

MultiIndex([('Fri',  'Lunch'),
            ('Fri', 'Dinner'),
            ('Sat',  'Lunch'),
            ('Sat', 'Dinner'),
            ('Sun',  'Lunch'),
            ('Sun', 'Dinner')],
           )

In [13]:
df.stack(future_stack=True)
#> 가장 마지막 레벨의 컬럼 이름들을 행 인덱스로 변환. columns ---> rows
#> level=-1 (기본값)

Fri  Sat  Sun
0 Lunch     1    3    5
  Dinner    2    4    6
1 Lunch     7    9   11
  Dinner    8   10   12

In [16]:
df.stack(level=0, future_stack=True)
#> 첫번째 레벨의 컬럼 이름들이 행 인덱스로 변환. columns ---> rows.

Lunch  Dinner
0 Fri      1       2
  Sat      3       4
  Sun      5       6
1 Fri      7       8
  Sat      9      10
  Sun     11      12

# pivot vs melt

In [18]:
df = pd.DataFrame(data={
    'time': ['Lunch'] * 3 + ['Dinner'] * 3,
    'day': ['Fri', 'Sat', 'Sun'] * 2,
    'tip': np.arange(1, 7),
    'total_bill': np.arange(10, 70, 10),
})
df

,time,day,tip,total_bill
0,Lunch,Fri,1,10
1,Lunch,Sat,2,20
2,Lunch,Sun,3,30
3,Dinner,Fri,4,40
4,Dinner,Sat,5,50
5,Dinner,Sun,6,60


## `pandas.DataFrame.pivot()`

*   카테고리(범주) 타입 컬럼의 값들을 컬럼 이름 또는 행 인덱스로 변환.
*   파라미터:
    *   `columns`: pivoting된 데이터프레임에서 컬럼 이름으로 사용하기 위한 변수 이름(들).
    *   `index`: pivoting된 데이터프레임에서 행 인덱스로 사용하기 위한 변수 이름(들).
    *   `values`: pivoting된 데이터프레임에서 각 셀을 채우기 위한 값들을 가지고 있는 변수 이름(들).


In [20]:
df.pivot(columns='day', index='time', values='tip')

day,Fri,Sat,Sun
time,,,
Dinner,4,5,6
Lunch,1,2,3


In [22]:
df.pivot(columns='time', index='day', values='tip')

time,Dinner,Lunch
day,,
Fri,4,1
Sat,5,2
Sun,6,3


In [26]:
df.pivot(columns='day', index='time', values=['tip', 'total_bill'])

tip         total_bill        
day    Fri Sat Sun        Fri Sat Sun
time                                 
Dinner   4   5   6         40  50  60
Lunch    1   2   3         10  20  30

In [27]:
df.pivot(columns='time', index='day', values=['tip', 'total_bill'])

tip       total_bill      
time Dinner Lunch     Dinner Lunch
day                               
Fri       4     1         40    10
Sat       5     2         50    20
Sun       6     3         60    30

## `pandas.DataFrame.melt()`

*   파라미터:
    *   `id_vars`: melting될 때 컬럼으로 유지할 변수 이름(들).
        *   `id_vars`에 설정하지 않은 변수 이름들은 **variable** 컬럼의 값들로 melting됨.
        *   `id_vars`에 설정하지 않은 변수의 값들은 **value** 컬럼의 값들로 melting됨.
    *   `var_name`: variable 컬럼 이름을 대체할 문자열.
    *   `value_name`: value 컬럼 이름을 대체할 문자열.


In [29]:
df = pd.DataFrame(data={
    'gender': ['Female', 'Male'],
    'lunch': [1, 5],
    'dinner': [10, 20],
})
df

,gender,lunch,dinner
0,Female,1,10
1,Male,5,20


In [30]:
df.melt(id_vars='gender')

,gender,variable,value
0,Female,lunch,1
1,Male,lunch,5
2,Female,dinner,10
3,Male,dinner,20


In [34]:
df.melt(id_vars='gender', var_name='time', value_name='size')

,gender,time,size
0,Female,lunch,1
1,Male,lunch,5
2,Female,dinner,10
3,Male,dinner,20


# Pivot Table

`pandas.DataFrame.pivot_table()`

*   groupby 기능과 통계 함수 적용 결과를 unstack하는 함수.
*   파라미터:
    *   `values`: 집계(통계) 함수를 적용할 값들을 가지고 있는 변수 이름(들).
    *   `index`: 피벗 테이블의 인덱스로 사용할 값들을 가지고 있는 변수 이름(들).
    *   `columns`: 피벗 테이블의 컬럼 이름으로 사용할 값들을 가지고 있는 변수 이름(들).
    *   `aggfunc`: aggregation function(집계 함수). 기본값은 'mean'.


In [35]:
tips = sns.load_dataset(name='tips')

In [36]:
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


## 성별 영수증 금액의 평균

In [41]:
result = tips.groupby(by='sex', observed=True).total_bill.mean()
result

,total_bill
sex,
Male,20.744076
Female,18.056897


In [50]:
tips.pivot_table(values='total_bill', index='sex', observed=True)  # aggfunc='mean' 기본값.

,total_bill
sex,
Male,20.744076
Female,18.056897


In [51]:
tips.pivot_table(values='total_bill', columns='sex', observed=True)

sex,Male,Female
total_bill,20.744076,18.056897


## 성별, 흡연여부별 팁 평균

In [53]:
result = tips.groupby(by=['sex', 'smoker'], observed=True).tip.mean()
result

sex     smoker
Male    Yes       3.051167
        No        3.113402
Female  Yes       2.931515
        No        2.773519
Name: tip, dtype: float64

In [54]:
result.unstack()

smoker,Yes,No
sex,,
Male,3.051167,3.113402
Female,2.931515,2.773519


In [55]:
result.unstack(level=0)

sex,Male,Female
smoker,,
Yes,3.051167,2.931515
No,3.113402,2.773519


In [56]:
tips.pivot_table(values='tip', index=['sex', 'smoker'], observed=True)

tip
sex    smoker          
Male   Yes     3.051167
       No      3.113402
Female Yes     2.931515
       No      2.773519

In [57]:
tips.pivot_table(values='tip', index='sex', columns='smoker', observed=True)

smoker,Yes,No
sex,,
Male,3.051167,3.113402
Female,2.931515,2.773519


In [58]:
tips.pivot_table(values='tip', index='smoker', columns='sex', observed=True)

sex,Male,Female
smoker,,
Yes,3.051167,2.931515
No,3.113402,2.773519


## 성별 팁과 영수증의 평균

In [67]:
tips.groupby(by='sex', observed=True)[['tip', 'total_bill']].mean()

,tip,total_bill
sex,,
Male,3.089618,20.744076
Female,2.833448,18.056897


In [68]:
tips.pivot_table(values=['tip', 'total_bill'], index='sex', observed=True)

,tip,total_bill
sex,,
Male,3.089618,20.744076
Female,2.833448,18.056897


In [69]:
tips.pivot_table(values=['tip', 'total_bill'], columns='sex', observed=True)

sex,Male,Female
tip,3.089618,2.833448
total_bill,20.744076,18.056897


## 성별 흡연여부별 팁과 영수증 평균

In [73]:
tips.groupby(by=['sex', 'smoker'], observed=True)[['tip', 'total_bill']].mean()

tip  total_bill
sex    smoker                      
Male   Yes     3.051167   22.284500
       No      3.113402   19.791237
Female Yes     2.931515   17.977879
       No      2.773519   18.105185

In [74]:
tips.pivot_table(values=['tip', 'total_bill'], index=['sex', 'smoker'], observed=True)

tip  total_bill
sex    smoker                      
Male   Yes     3.051167   22.284500
       No      3.113402   19.791237
Female Yes     2.931515   17.977879
       No      2.773519   18.105185

In [75]:
tips.pivot_table(values=['tip', 'total_bill'], index='sex', columns='smoker', observed=True)

tip           total_bill           
smoker       Yes        No        Yes         No
sex                                             
Male    3.051167  3.113402  22.284500  19.791237
Female  2.931515  2.773519  17.977879  18.105185

In [76]:
tips.pivot_table(values=['tip', 'total_bill'], index='smoker', columns='sex', observed=True)

tip           total_bill           
sex         Male    Female       Male     Female
smoker                                          
Yes     3.051167  2.931515  22.284500  17.977879
No      3.113402  2.773519  19.791237  18.105185

## 성별, 요일별, 시간별 팁의 평균

In [79]:
tips.groupby(by=['sex', 'day', 'time'], observed=True).tip.mean()

sex     day   time  
Male    Thur  Lunch     2.980333
        Fri   Lunch     1.900000
              Dinner    3.032857
        Sat   Dinner    3.083898
        Sun   Dinner    3.220345
Female  Thur  Lunch     2.561935
              Dinner    3.000000
        Fri   Lunch     2.745000
              Dinner    2.810000
        Sat   Dinner    2.801786
        Sun   Dinner    3.367222
Name: tip, dtype: float64

In [80]:
tips.pivot_table(values='tip', index=['sex', 'day', 'time'], observed=True)

tip
sex    day  time            
Male   Thur Lunch   2.980333
       Fri  Lunch   1.900000
            Dinner  3.032857
       Sat  Dinner  3.083898
       Sun  Dinner  3.220345
Female Thur Lunch   2.561935
            Dinner  3.000000
       Fri  Lunch   2.745000
            Dinner  2.810000
       Sat  Dinner  2.801786
       Sun  Dinner  3.367222

In [81]:
tips.pivot_table(values='tip', index='day', columns=['sex', 'time'], observed=True)

sex       Male              Female          
time     Lunch    Dinner     Lunch    Dinner
day                                         
Thur  2.980333       NaN  2.561935  3.000000
Fri   1.900000  3.032857  2.745000  2.810000
Sat        NaN  3.083898       NaN  2.801786
Sun        NaN  3.220345       NaN  3.367222

In [82]:
tips.pivot_table(values='tip', index=['sex', 'time'], columns='day', observed=True)

day                Thur       Fri       Sat       Sun
sex    time                                          
Male   Lunch   2.980333  1.900000       NaN       NaN
       Dinner       NaN  3.032857  3.083898  3.220345
Female Lunch   2.561935  2.745000       NaN       NaN
       Dinner  3.000000  2.810000  2.801786  3.367222

## 성별 팁의 평균, 최솟값, 중앙값, 최댓값

In [86]:
tips.groupby(by='sex', observed=True).tip.agg(['mean', 'min', 'median', 'max'])
# agg: aggregate(집계하다)

,mean,min,median,max
sex,,,,
Male,3.089618,1.0,3.00,10.0
Female,2.833448,1.0,2.75,6.5


In [87]:
tips.pivot_table(values='tip', index='sex', aggfunc=['mean', 'min', 'median', 'max'],
                 observed=True)

,mean,min,median,max
,tip,tip,tip,tip
sex,,,,
Male,3.089618,1.0,3.00,10.0
Female,2.833448,1.0,2.75,6.5


In [88]:
tips.pivot_table(values='tip', columns='sex', aggfunc=['mean', 'min', 'median', 'max'],
                 observed=True)

mean            min        median          max       
sex      Male    Female Male Female   Male Female  Male Female
tip  3.089618  2.833448  1.0    1.0    3.0   2.75  10.0    6.5

## 성별, 요일별 팁의 평균, 최솟값, 최댓값

In [90]:
tips.groupby(by=['sex', 'day'], observed=True).tip.aggregate(['mean', 'min', 'max'])

mean   min    max
sex    day                        
Male   Thur  2.980333  1.44   6.70
       Fri   2.693000  1.50   4.73
       Sat   3.083898  1.00  10.00
       Sun   3.220345  1.32   6.50
Female Thur  2.575625  1.25   5.17
       Fri   2.781111  1.00   4.30
       Sat   2.801786  1.00   6.50
       Sun   3.367222  1.01   5.20

In [91]:
tips.pivot_table(values='tip', index=['sex', 'day'], aggfunc=['mean', 'min', 'max'],
                 observed=True)

mean   min    max
                  tip   tip    tip
sex    day                        
Male   Thur  2.980333  1.44   6.70
       Fri   2.693000  1.50   4.73
       Sat   3.083898  1.00  10.00
       Sun   3.220345  1.32   6.50
Female Thur  2.575625  1.25   5.17
       Fri   2.781111  1.00   4.30
       Sat   2.801786  1.00   6.50
       Sun   3.367222  1.01   5.20

In [92]:
tips.pivot_table(values='tip', index='day', columns='sex', aggfunc=['mean', 'min', 'max'],
                 observed=True)

mean             min           max       
sex       Male    Female  Male Female   Male Female
day                                                
Thur  2.980333  2.575625  1.44   1.25   6.70   5.17
Fri   2.693000  2.781111  1.50   1.00   4.73   4.30
Sat   3.083898  2.801786  1.00   1.00  10.00   6.50
Sun   3.220345  3.367222  1.32   1.01   6.50   5.20

## 성별, 흡연여부별, 요일별, 시간별 팁의 평균

In [94]:
tips.groupby(by=['sex', 'smoker', 'day', 'time'], observed=True).tip.mean()

sex     smoker  day   time  
Male    Yes     Thur  Lunch     3.058000
                Fri   Lunch     1.900000
                      Dinner    3.246000
                Sat   Dinner    2.879259
                Sun   Dinner    3.521333
        No      Thur  Lunch     2.941500
                Fri   Dinner    2.500000
                Sat   Dinner    3.256563
                Sun   Dinner    3.115349
Female  Yes     Thur  Lunch     2.990000
                Fri   Lunch     2.660000
                      Dinner    2.700000
                Sat   Dinner    2.868667
                Sun   Dinner    3.500000
        No      Thur  Lunch     2.437083
                      Dinner    3.000000
                Fri   Lunch     3.000000
                      Dinner    3.250000
                Sat   Dinner    2.724615
                Sun   Dinner    3.329286
Name: tip, dtype: float64

In [95]:
tips.pivot_table(values='tip', index=['sex', 'smoker', 'day', 'time'], observed=True)

tip
sex    smoker day  time            
Male   Yes    Thur Lunch   3.058000
              Fri  Lunch   1.900000
                   Dinner  3.246000
              Sat  Dinner  2.879259
              Sun  Dinner  3.521333
       No     Thur Lunch   2.941500
              Fri  Dinner  2.500000
              Sat  Dinner  3.256563
              Sun  Dinner  3.115349
Female Yes    Thur Lunch   2.990000
              Fri  Lunch   2.660000
                   Dinner  2.700000
              Sat  Dinner  2.868667
              Sun  Dinner  3.500000
       No     Thur Lunch   2.437083
                   Dinner  3.000000
              Fri  Lunch   3.000000
                   Dinner  3.250000
              Sat  Dinner  2.724615
              Sun  Dinner  3.329286

In [96]:
tips.pivot_table(values='tip', index=['day', 'time'], columns=['sex', 'smoker'], observed=True)

sex              Male              Female          
smoker            Yes        No       Yes        No
day  time                                          
Thur Lunch   3.058000  2.941500  2.990000  2.437083
     Dinner       NaN       NaN       NaN  3.000000
Fri  Lunch   1.900000       NaN  2.660000  3.000000
     Dinner  3.246000  2.500000  2.700000  3.250000
Sat  Dinner  2.879259  3.256563  2.868667  2.724615
Sun  Dinner  3.521333  3.115349  3.500000  3.329286

## 성별, 흡연여부별, 요일별, 시간별 팁과 영수증의 평균, 중앙값

In [98]:
tips.groupby(by=['sex', 'smoker', 'day', 'time'], observed=True)[['tip', 'total_bill']].agg(['mean', 'median'])

tip        total_bill        
                               mean median       mean  median
sex    smoker day  time                                      
Male   Yes    Thur Lunch   3.058000  2.780  19.171000  17.645
              Fri  Lunch   1.900000  1.920  11.386667  12.160
                   Dinner  3.246000  3.000  25.892000  27.280
              Sat  Dinner  2.879259  3.000  21.837778  20.290
              Sun  Dinner  3.521333  3.500  26.141333  23.330
       No     Thur Lunch   2.941500  2.405  18.486500  16.975
              Fri  Dinner  2.500000  2.500  17.475000  17.475
              Sat  Dinner  3.256563  2.860  19.929063  17.870
              Sun  Dinner  3.115349  3.000  20.403256  19.490
Female Yes    Thur Lunch   2.990000  2.500  19.218571  16.400
              Fri  Lunch   2.660000  2.500  13.260000  13.420
                   Dinner  2.700000  2.750  12.200000  13.365
              Sat  Dinner  2.868667  2.500  20.266667  22.120
              Sun  Dinner  3.500000  3.500  16.540000  17.830
       No     Thur Lunch   2.437083  2.000  15.899167  13.290
                   Dinner  3.000000  3.000  18.780000  18.780
              Fri  Lunch   3.000000  3.000  15.980000  15.980
                   Dinner  3.250000  3.250  22.750000  22.750
              Sat  Dinner  2.724615  2.750  19.003846  17.070
              Sun  Dinner  3.329286  3.500  20.824286  17.150

In [100]:
tips.pivot_table(values=['tip', 'total_bill'],
                 index=['day', 'time'],
                 columns=['sex', 'smoker'],
                 aggfunc=['mean', 'median'],
                 observed=True)

mean                                                      \
                  tip                               total_bill              
sex              Male              Female                 Male              
smoker            Yes        No       Yes        No        Yes         No   
day  time                                                                   
Thur Lunch   3.058000  2.941500  2.990000  2.437083  19.171000  18.486500   
     Dinner       NaN       NaN       NaN  3.000000        NaN        NaN   
Fri  Lunch   1.900000       NaN  2.660000  3.000000  11.386667        NaN   
     Dinner  3.246000  2.500000  2.700000  3.250000  25.892000  17.475000   
Sat  Dinner  2.879259  3.256563  2.868667  2.724615  21.837778  19.929063   
Sun  Dinner  3.521333  3.115349  3.500000  3.329286  26.141333  20.403256   

                                  median                                 \
                                     tip                     total_bill   
sex             Female              Male        Female             Male   
smoker             Yes         No    Yes     No    Yes    No        Yes   
day  time                                                                 
Thur Lunch   19.218571  15.899167   2.78  2.405   2.50  2.00     17.645   
     Dinner        NaN  18.780000    NaN    NaN    NaN  3.00        NaN   
Fri  Lunch   13.260000  15.980000   1.92    NaN   2.50  3.00     12.160   
     Dinner  12.200000  22.750000   3.00  2.500   2.75  3.25     27.280   
Sat  Dinner  20.266667  19.003846   3.00  2.860   2.50  2.75     20.290   
Sun  Dinner  16.540000  20.824286   3.50  3.000   3.50  3.50     23.330   

                                    
                                    
sex                  Female         
smoker           No     Yes     No  
day  time                           
Thur Lunch   16.975  16.400  13.29  
     Dinner     NaN     NaN  18.78  
Fri  Lunch      NaN  13.420  15.98  
     Dinner  17.475  13.365  22.75  
Sat  Dinner  17.870  22.120  17.07  
Sun  Dinner  19.490  17.830  17.15